# Practical Session 5: Perceptron Learning Algorithm and Linear Classification

This notebook implements the perceptron algorithm from scratch and comments each major
step so students can follow the full classification workflow directly in Python.


In [ ]:
# Import NumPy for matrix-style sample handling.
import numpy as np
# Import pandas for experiment summary tables.
import pandas as pd
# Import Matplotlib for data and boundary plots.
import matplotlib.pyplot as plt

# Keep arrays readable in printed output.
np.set_printoptions(precision=4, suppress=True)
# Fix the seed for the synthetic-data experiments later in the notebook.
rng = np.random.default_rng(42)

# Store the sample names to annotate the plots.
names = np.array(["Mueller", "Kroos", "Reus", "Gomez", "Goetze"])
# Store the two input features: goals and PCR value.
features = np.array(
    [
        [10.0, 0.1],
        [2.0, 0.7],
        [6.0, 0.6],
        [8.0, 0.1],
        [8.0, 0.4],
    ]
)
# Encode the class labels of the first four samples as +1 and -1.
labels = np.array([1, -1, 1, -1])

# Use the first four samples for training, as requested in the script.
X_train = features[:4]
# Reserve the fifth sample for the final prediction task.
x_goetze = features[4]


## Task 1: Data preparation and visualization

We start by plotting the two classes in the feature plane to build geometric intuition.


In [ ]:
# Create a scatter plot for the positive class.
plt.figure(figsize=(6, 4))
plt.scatter(X_train[labels == 1, 0], X_train[labels == 1, 1], label="Class +1", s=80)
# Create a second scatter plot for the negative class.
plt.scatter(X_train[labels == -1, 0], X_train[labels == -1, 1], label="Class -1", s=80, marker="x")
# Annotate each training sample with its player name.
for idx, name in enumerate(names[:4]):
    plt.annotate(name, (X_train[idx, 0], X_train[idx, 1]), textcoords="offset points", xytext=(5, 5))
plt.xlabel("Goals")
plt.ylabel("PCR value")
plt.title("Training data for the perceptron")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Tasks 2 and 3: Perceptron implementation, prediction, and boundary plot

The perceptron uses a bias term, loops over the training samples, and updates the
weight vector whenever a sample is misclassified.


In [ ]:
def augment_with_bias(X):
    # Prepend a column of ones so the bias becomes part of the weight vector.
    # TODO: Add a bias column of ones in front of the feature matrix.
    return ...


def perceptron_train(X, y, alpha=1.0, w0=None, max_epochs=1000, order=None):
    # Add the bias column once at the beginning.
    X_aug = augment_with_bias(X)
    # Start from the zero vector unless an initial weight vector is provided.
    w = np.zeros(X_aug.shape[1]) if w0 is None else np.array(w0, dtype=float).copy()
    # Collect per-epoch diagnostics for later inspection.
    update_history = []

    for epoch in range(max_epochs):
        errors = 0
        # Use the natural ordering unless a custom permutation is given.
        indices = np.arange(len(y)) if order is None else np.array(order)
        for idx in indices:
            # The signed margin tells us whether the current sample is classified correctly.
            margin = y[idx] * np.dot(w, X_aug[idx])
            if margin <= 0:
                # Apply the classical perceptron update when the sample is wrong.
                # TODO: Apply the perceptron update rule.
                ...
                errors += 1
        # Compute training predictions after the current epoch.
        predictions = np.sign(X_aug @ w)
        predictions[predictions == 0] = 1
        error_rate = np.mean(predictions != y)
        update_history.append(
            {"epoch": epoch + 1, "updates": errors, "error_rate": error_rate, "w": w.copy()}
        )
        # Stop early when no sample was misclassified in the current epoch.
        if errors == 0:
            return w, True, pd.DataFrame(update_history)
    return w, False, pd.DataFrame(update_history)


# Train the perceptron on the original data set.
w_star, converged, history_df = perceptron_train(X_train, labels, alpha=1.0, max_epochs=1000)
# Augment the training matrix so predictions use the learned bias as well.
X_train_aug = augment_with_bias(X_train)
train_predictions = np.sign(X_train_aug @ w_star)
train_predictions[train_predictions == 0] = 1
# Predict the class of the new sample Goetze.
goetze_prediction = int(np.sign(np.dot(np.r_[1.0, x_goetze], w_star)) or 1)

print("Converged:", converged)
print("Learned weight vector [bias, w1, w2]:", w_star)
print("Training predictions:", train_predictions.astype(int))
print("Predicted class for Goetze:", goetze_prediction)
display(history_df.head())


In [ ]:
def plot_boundary(ax, w, X, y, title):
    # Plot the positive class.
    ax.scatter(X[y == 1, 0], X[y == 1, 1], label="Class +1", s=80)
    # Plot the negative class.
    ax.scatter(X[y == -1, 0], X[y == -1, 1], label="Class -1", s=80, marker="x")
    # Create a horizontal grid of x-values for the decision line.
    x_values = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    if abs(w[2]) > 1e-12:
        # Solve w0 + w1*x + w2*y = 0 for y to draw the decision boundary.
        y_values = -(w[0] + w[1] * x_values) / w[2]
        ax.plot(x_values, y_values, color="black", label="Decision boundary")
    ax.set_xlabel("Goals")
    ax.set_ylabel("PCR value")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)


# Plot the learned separator together with the training samples.
fig, ax = plt.subplots(figsize=(6, 4))
plot_boundary(ax, w_star, X_train, labels, "Perceptron decision boundary")
# Highlight Goetze as the unlabeled sample to be classified.
ax.scatter(x_goetze[0], x_goetze[1], c="gold", edgecolor="black", s=120, label="Goetze")
plt.show()


## Task 4: Influence of learning rate, initialization, and ordering

The next experiment runs the perceptron under several configurations and records the
convergence behavior and final parameter vector.


In [ ]:
# Define several configurations that differ in learning rate, initialization, or order.
configurations = [
    {"name": "alpha=0.1, zero init", "alpha": 0.1, "w0": None, "order": None},
    {"name": "alpha=1.0, zero init", "alpha": 1.0, "w0": None, "order": None},
    {"name": "alpha=0.5, custom init", "alpha": 0.5, "w0": [0.2, -0.1, 0.3], "order": None},
    {"name": "alpha=1.0, reversed order", "alpha": 1.0, "w0": None, "order": [3, 2, 1, 0]},
]

config_rows = []
for config in configurations:
    # Train the perceptron under the current configuration.
    # TODO: Train the perceptron for the current configuration.
    w_final, did_converge, run_history = ...
    config_rows.append(
        {
            "configuration": config["name"],
            "converged": did_converge,
            "total_updates": int(run_history["updates"].sum()),
            "final_weights": np.round(w_final, 3).tolist(),
        }
    )

# Display the comparison table across all tested settings.
display(pd.DataFrame(config_rows))


## Task 5: Nonseparable data and practical limits

To violate the separability assumption, we make two points identical while keeping their
labels different. This prevents perfect linear separation.


In [ ]:
# Copy the original features and create a conflicting duplicate point.
X_nonseparable = X_train.copy()
X_nonseparable[3] = X_nonseparable[0]

# Train the perceptron on the modified data set.
w_nonsep, converged_nonsep, nonsep_history = perceptron_train(
    X_nonseparable, labels, alpha=1.0, max_epochs=80
)
print("Converged on modified nonseparable data:", converged_nonsep)

# Plot the training error over epochs to show the lack of convergence.
plt.figure(figsize=(6, 4))
plt.plot(nonsep_history["epoch"], nonsep_history["error_rate"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training error rate")
plt.title("Perceptron behavior on a nonseparable data set")
plt.grid(True, alpha=0.3)
plt.show()


## Task 6: Random synthetic data

We finish with one linearly separable and one deliberately nonseparable synthetic data
set to compare convergence behavior and final accuracy.


In [ ]:
def synthetic_dataset(separable=True, n_per_class=20, seed=0):
    # Use a local random generator so each call is reproducible on its own.
    local_rng = np.random.default_rng(seed)
    # Sample a positive cluster around one center.
    positive = local_rng.normal(loc=[2.0, 2.0], scale=0.5, size=(n_per_class, 2))
    # Sample a negative cluster around another center.
    negative = local_rng.normal(loc=[-2.0, -1.5], scale=0.5, size=(n_per_class, 2))
    X = np.vstack([positive, negative])
    y = np.r_[np.ones(n_per_class, dtype=int), -np.ones(n_per_class, dtype=int)]
    if not separable:
        # Overwrite the positive cluster by a broad overlapping cloud.
        overlap = local_rng.normal(loc=[0.0, 0.0], scale=1.2, size=(n_per_class, 2))
        X[:n_per_class] = overlap
        # Flip a few labels to create additional contradictions.
        # TODO: Flip a few labels to make the synthetic data nonseparable.
        ...
    return X, y


# Generate one separable and one nonseparable synthetic data set.
X_sep, y_sep = synthetic_dataset(separable=True, seed=1)
X_nonsep, y_nonsep = synthetic_dataset(separable=False, seed=2)

# Train the same implementation on both data sets.
w_sep, conv_sep, hist_sep = perceptron_train(X_sep, y_sep, alpha=1.0, max_epochs=200)
w_nonsep2, conv_nonsep2, hist_nonsep2 = perceptron_train(X_nonsep, y_nonsep, alpha=1.0, max_epochs=200)

# Compare convergence and final training accuracy.
synthetic_summary = pd.DataFrame(
    [
        {
            "data_set": "separable",
            "converged": conv_sep,
            "final_accuracy": 1 - hist_sep["error_rate"].iloc[-1],
        },
        {
            "data_set": "nonseparable",
            "converged": conv_nonsep2,
            "final_accuracy": 1 - hist_nonsep2["error_rate"].iloc[-1],
        },
    ]
)
display(synthetic_summary)
